In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
import random
from matplotlib import colors
import os, glob
from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import LinearSegmentedColormap
import colorsys
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.legend_handler import HandlerBase
from matplotlib.collections import LineCollection
from PIL import Image

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
def log_fit(x, a, b):
    return a + b * np.log(x)

In [ ]:
def obtain_ratio_08_with_uncertainty(path, column_name):
    files = glob.glob(path)
    df_list = []
    
    # --- Data loading (unchanged) ---
    for file in files:
        tmp_df = pd.read_csv(file)
        country = file.split('/')[-5]
        city = file.split('/')[-4]
        label_sdg_file = f"../../data/processed/0labels/{country}.csv"
        labels_sdg = pd.read_csv(label_sdg_file)
        tmp_df = pd.merge(tmp_df, labels_sdg[['ID', 'SDG']], left_on='target', right_on='ID')
        tmp_df['country'] = country
        tmp_df['city'] = city
        df_list.append(tmp_df)
    
    df = pd.concat(df_list, ignore_index=True)
    df['country_city'] = df['country'] + '_' + df['city']
    df.drop(['target', 'ID', 'country', 'city', 'SDG'], axis=1, inplace=True)
    
    # Aggregate data
    grouped = df.groupby(['ratio', 'country_city']).mean().reset_index()
    city_list = grouped['country_city'].unique().tolist()

    city_results = []
    target_r2 = 0.8  # target R^2

    for city in city_list:
        city_df = grouped[grouped['country_city'] == city]
        
        x_data = city_df['ratio']
        y_data = city_df['all_r2']
        
        try:
            # 1. Fit.
            params_r2, covariance = curve_fit(log_fit, x_data, y_data)
            a, b = params_r2
            
            # 2. Point estimate.
            # Formula: x = exp((0.8 - a) / b).
            exponent = (target_r2 - a) / b
            x_08 = np.exp(exponent)
            
            # 3. Uncertainty (delta method).
            # Extract covariance matrix elements.
            var_a = covariance[0, 0]  # variance of a
            var_b = covariance[1, 1]  # variance of b
            cov_ab = covariance[0, 1] # covariance of a and b
            
            # Compute partial derivatives (Jacobian).
            # dx/da = -x / b
            dx_da = -x_08 / b
            # dx/db = -x * (0.8 - a) / b^2
            dx_db = -x_08 * (target_r2 - a) / (b**2)
            
            # Variance of x_08.
            var_x = (dx_da**2 * var_a) + (dx_db**2 * var_b) + (2 * dx_da * dx_db * cov_ab)
            
            # Standard error.
            std_x = np.sqrt(var_x)
            
            # 95% confidence interval (1.96 * sigma).
            ci_lower = x_08 - 1.96 * std_x
            ci_upper = x_08 + 1.96 * std_x
            
            # Store results.
            city_results.append((city, x_08, std_x, ci_lower, ci_upper))
            
            # print(f"{city}: Ratio={x_08:.4f} ± {std_x:.4f}")

        except Exception as e:
            print(f"Fit failed for {city}: {e}")
            city_results.append((city, np.nan, np.nan, np.nan, np.nan))

    # Build the results DataFrame.
    # Columns: mean, std, CI lower, CI upper.
    columns = ['city', column_name, f'{column_name}_std', f'{column_name}_lower', f'{column_name}_upper']
    city_08_df = pd.DataFrame(city_results, columns=columns)
    for column in columns[1:]:
        city_08_df[column] = city_08_df[column] * 100
    
    return city_08_df

In [ ]:
ours_df = obtain_ratio_08_with_uncertainty('../../data/regression_outputs/regmodels_spatial_self/Sampling_kcenter/*/*/Fuse/Token_Concat_spatial_self_Spatial/results.csv', 'Ours')
imagenet_df = obtain_ratio_08_with_uncertainty('../../data/regression_outputs/regmodels_spatial_self/Ratio/*/*/Fuse/Token_Concat_spatial_self_Spatial_Random/results.csv', 'Random')

df  = pd.merge(ours_df, imagenet_df, on='city', how='outer')
df['country'] = df['city'].apply(lambda x: x.split('_')[0])
def get_city_name(name):
    if name == 'France_All':
        return 'French Cities'
    elif name == 'Portugal_All':
        return 'Portuguese Cities'
    elif name == 'Brazil_BeloHorizonte':
        return 'Belo Horizonte'
    elif name == 'Brazil_PortoAlegre':
        return 'Porto Alegre'
    elif name == 'US_LosAngeles':
        return 'Los Angeles'
    elif name == 'US_NewYork':
        return 'New York'
    elif name == 'US_SanFrancisco':
        return 'San Francisco'
    elif name == 'Brazil_RiodeJaneiro':
        return 'Rio de Janeiro'
    elif name == 'China_HongKong':
        return 'Hong Kong'
    else:
        return name.split('_')[1]
df['city_name'] = df['city'].apply(get_city_name)
df.sort_values(by=['Ours'], inplace=True)
df

In [ ]:
# Custom legend handler that renders flag icons.
class ImageHandler(HandlerBase):
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        img = Image.open(orig_handle['path']).convert('RGBA')
        img = img.resize((int(width * 2), int(height)), Image.Resampling.LANCZOS)
        oi = OffsetImage(img, zoom=1)
        ab = AnnotationBbox(oi, (xdescent + width / 2, ydescent + height / 2),
                            frameon=False, xycoords='axes points', boxcoords="offset points")
        return [ab]

def darken_color(hex_color, factor=0.7):
    # HEX -> RGB.
    rgb = tuple(int(hex_color.lstrip('#')[i:i+2], 16) / 255 for i in (0, 2, 4))
    # Convert to HLS and lower lightness.
    h, l, s = colorsys.rgb_to_hls(*rgb)
    l = max(0, l * factor)  # smaller factor -> darker
    # Convert back to RGB then HEX.
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

In [ ]:
def create_gradient_segments(x_start, x_end, y, cmap, num_segments=50):
    """
    Pre-compute all line segments to avoid the plot loop.
    """
    segments = []
    colors = []
    
    # Just generate a color sequence in [0, 1].
    color_indices = np.linspace(0, 1, num_segments)
    rgba_colors = cmap(color_indices)
    
    for i in range(len(x_start)):
        # Build line segments for each row.
        xs = np.linspace(x_start[i], x_end[i], num_segments + 1)
        ys = np.full_like(xs, y[i])
        
        # Reshape points to segment form [(x0, y0), (x1, y1)].
        points = np.array([xs, ys]).T.reshape(-1, 1, 2)
        segs = np.concatenate([points[:-1], points[1:]], axis=1)
        
        segments.extend(segs)
        colors.extend(rgba_colors)
        
    return segments, colors

In [ ]:
# --- Configuration ---
fig_metas = {
    'Ours': {'color': '#a0d3d5', 'zorder': 10},
    'Random': {'color': '#e2a58b', 'zorder': 5}
}

flag_paths = {
    "US": "../../data/figure_assets/country_flag/us.png",
    "Australia": "../../data/figure_assets/country_flag/au.png",
    "Brazil": "../../data/figure_assets/country_flag/br.png",
    "China": "../../data/figure_assets/country_flag/cn.png",
    "France": "../../data/figure_assets/country_flag/fr.png",
    "Portugal": "../../data/figure_assets/country_flag/pt.png",
    "Nigeria": "../../data/figure_assets/country_flag/ng.png",
}

# --- Plot start ---
fig, ax = plt.subplots(figsize=(6, 10)) # height scales with #cities

# 1. Gradient connector lines (LineCollection for speed).
cmap1 = LinearSegmentedColormap.from_list('custom', [fig_metas['Ours']['color'], fig_metas['Random']['color']])
y_values = np.arange(len(df))

# Build segments.
segments, seg_colors = create_gradient_segments(
    df['Ours'].values, 
    df['Random'].values, 
    y_values, 
    cmap1, 
    num_segments=50 # 50 segments is smooth enough
)

# Build the collection and attach to the axis.
lc = LineCollection(segments, colors=seg_colors, linewidths=2.5, alpha=0.8, linestyle='solid', zorder=1)
ax.add_collection(lc)

# 2. Scatter + summary region.
y_min, y_max = -1, len(df) # default y-range

for k in fig_metas.keys():
    # Scatter.
    ax.scatter(
        df[k], 
        y_values, 
        color=fig_metas[k]['color'], 
        s=180, 
        edgecolor=darken_color(fig_metas[k]['color'], 0.7),
        linewidth=1.5,
        zorder=fig_metas[k]['zorder'],
        label=k
    )
    
    # Summary stats.
    mean_val = df[k].mean()
    std_val = df[k].std()
    lower, upper = mean_val - 1.96 * std_val, mean_val + 1.96 * std_val
    dark_color = darken_color(fig_metas[k]['color'], 0.6)
    
    # Shaded rectangle (full height).
    # Note: uses ax.get_ylim, or just draw a tall rectangle and clip.
    rect = Rectangle(
        (lower, -2), # y starts at -2 to fully cover the bottom
        upper - lower, 
        len(df) + 4, # height
        color=fig_metas[k]['color'], 
        alpha=0.15, 
        zorder=0,
        linewidth=0
    )
    ax.add_patch(rect)
    
    # Vertical line at the mean.
    ax.axvline(x=mean_val, color=darken_color(fig_metas[k]['color'], 0.8), 
               linestyle='-', linewidth=1.5, zorder=0, alpha=0.6)

    # Label Avg. and Std. once, positioned by data.
    # Slight y offset to avoid overlap.
    anno_y_base = -0.5 if k == 'Ours' else -1.5
    
    # E1. Annotate the mean.
    ax.text(
        mean_val, anno_y_base + 0.4, 
        f'{k} Avg.', 
        fontsize=12, fontweight='bold',
        color=dark_color,
        ha='center', va='bottom'
    )
    
    # E2. Horizontal bar for Std (95% CI).
    # Connect lower and upper with a horizontal line.
    ax.plot([lower, upper], [anno_y_base, anno_y_base], 
            color=dark_color, linewidth=1.5, clip_on=False, zorder=20)
    
    # Tick marks at the endpoints.
    tick_h = 0.3
    ax.plot([lower, lower], [anno_y_base - tick_h/2, anno_y_base + tick_h/2], 
            color=dark_color, linewidth=1.5, clip_on=False, zorder=20)
    ax.plot([upper, upper], [anno_y_base - tick_h/2, anno_y_base + tick_h/2], 
            color=dark_color, linewidth=1.5, clip_on=False, zorder=20)

    # E3. Label 'Std.'
    ax.text(
        mean_val, anno_y_base + 0.4, 
        r'$\pm$1.96 Std.', 
        fontsize=10, 
        color=dark_color,
        ha='center', va='top'
    )
    
    print(f'Mean: {mean_val:.2f}, Std: {std_val:.2f}, 95% CI: [{lower:.2f}, {upper:.2f}]')

# 3. Y-axis labels and flag icons.
ax.set_yticks(y_values)
ax.set_yticklabels(df['city_name'], fontsize=14)

# Flag parameters.
flag_zoom = 0.08 # scale factor for the flag image
x_offset_flag = -0.18 # negative -> left of the y-axis (axes coords)

for idx, (city_name, country) in enumerate(zip(df['city_name'], df['country'])):
    flag_path = flag_paths.get(country)
    
    if flag_path:
        try:
            img = Image.open(flag_path).convert('RGBA')
            # Preserve aspect ratio when scaling.
            img.thumbnail((100, 100), Image.Resampling.LANCZOS) 
            oi = OffsetImage(img, zoom=0.2) # zoom controls size
            
            # Key trick: use blended_transform_factory.
            # x in axes coords (0 = left axis), y in data coords (row index).
            # Flags stay fixed on the left regardless of x-axis range.
            ab = AnnotationBbox(
                oi,
                (0.11, idx), # (x_axes_value, y_data_value)
                xybox=(-45, 0), # 45 points left of the anchor
                xycoords=('axes fraction', 'data'),
                boxcoords="offset points",
                frameon=False
            )
            ax.add_artist(ab)
            
            # Shift the text labels to make room for flags.
            # matplotlib tick labels are hard to shift individually;
            # the usual workaround is more tick padding or moving the flag further out.
            
        except Exception as e:
            print(f"Error loading flag for {country}: {e}")

ax.invert_xaxis()
# 4. Style.
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False) # hide left spine
ax.spines['bottom'].set_visible(False)

# Add a light grid to aid comparison.
ax.grid(axis='x', color='gray', linestyle='--', alpha=0.2)
ax.set_axisbelow(True) # grid below data

# Set ranges.
ax.set_ylim(-2, len(df))
ax.set_xlabel('Average surveyed areas at overall $R^2$ = 0.8 (%)', fontsize=16)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', pad=24) # extra padding for flag icons

plt.tight_layout()
plt.savefig('../../data/figure_assets/fig1_city_points.svg', format='svg', bbox_inches='tight')
plt.show()